# Import Packages (Enhanced model)

In [ ]:
import time

# Start counting notebook running time
time_start = time.time()

import numpy as np
import tensorflow as tf
# Set all random seeds for the program (Python, NumPy, and TensorFlow)
tf.keras.utils.set_random_seed(1)

import pandas as pd
from  IPython.display import display, Image
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from keras.models import Sequential
from keras.layers import Dense, Flatten, LSTM, RepeatVector, TimeDistributed, Dropout
from keras.layers import Conv1D, MaxPooling1D
from keras import regularizers
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import sklearn
import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from time import strftime
current_time = strftime("%Y-%m-%d-%H-%M-%S")
print('current time: ', current_time)
print('numpy version: ', np.__version__)
print('tensorflow version: ', tf.__version__)
print('pandas version: ', pd.__version__)
print('scikit-learn version: ', sklearn.__version__)
print('keras version: ', keras.__version__)

In [ ]:
# Load the dataset
file_path = '/mnt/c/Users/xin37/github/Hybrid-Deep-Learning-DRL-Data-Center-Thermal-Management/data/TDC2_processed.csv' 

print("Loading and parsing dataset...")
df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)

print("Data loaded successfully! Shape:", df.shape)
print(df.head(3))

In [ ]:
def detect_outliers_iqr(df, cols):
    outlier_flags = pd.DataFrame(index=df.index)
    
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outlier_flags[col + '_outlier'] = ((df[col] < lower) | (df[col] > upper))
    
    return outlier_flags

num_cols = df.select_dtypes(include=["float64", "int64"]).columns
outliers = detect_outliers_iqr(df, num_cols)

print(f"\nOutlier summary (IQR method):\n",outliers.sum())

In [ ]:
import seaborn as sns

# Calculate the correlation matrix
corr_matrix = df.corr()

# 3. Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_matrix, 
    annot=True,              
    cmap='coolwarm',         
    fmt=".2f",               
    linewidths=0.5,          
    vmin=-1, vmax=1          
)

plt.title('Correlation Matrix', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Visualize sample data

In [ ]:
%%time

fig = plt.figure(figsize=(14, 6)) # Made slightly wider for 33,000+ rows

# Using the new target variable from the Kaggle dataset
plt.plot(df.index, df['T_Return'], color='blue', linewidth=0.5) 

plt.xlabel('Time (Date)')
plt.ylabel('Return Air Temperature (°C)')
plt.title('HVAC Return Air Temperature Over Time (30-Second Intervals)')
plt.grid(True) 
plt.show()

# Split a dataset into train/test sets

In [ ]:
from sklearn.preprocessing import MinMaxScaler

n_input  = 16        # Look back 8 mins (16 steps * 30 sec = 8 mins)
n_output = 1         # Predict next 30seconds

test_size = 2 * 60 * 24 * 7  # in 30-sec resolution, 1 week = 20160 steps

# Using the highly correlated features 
FEATURE_COLS = [
    'T_Return',
    'T_Supply',
    'T_Outdoor',
    'HVAC_Power_kW',
    'RH_Outdoor',
    'IT_Power_Total_kW' 
]

TARGET_COL = 'T_Return' 

# Create Sliding Windows
def create_sequences(X, y, n_input, n_output):
    X_seq, y_seq = [], []
    for i in range(len(X) - n_input - n_output + 1):
        X_seq.append(X[i : i + n_input])
        y_seq.append(y[i + n_input : i + n_input + n_output])
    return np.array(X_seq), np.array(y_seq).squeeze()

# Split and Scale Function
def prepare_data(data, test_size_steps):
    split_idx = len(data) - test_size_steps
    
    # Split into Train and Test sets
    train_df = data.iloc[:split_idx] 
    test_df = data.iloc[split_idx:] 
    
    # Scale Features
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(train_df[FEATURE_COLS]) 
    X_test_scaled = scaler.transform(test_df[FEATURE_COLS]) 
    
    target_scaler = MinMaxScaler()
    y_train_scaled = target_scaler.fit_transform(train_df[[TARGET_COL]])
    y_test_scaled  = target_scaler.transform(test_df[[TARGET_COL]])
    
    # Create Sequences
    X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, n_input, n_output)
    X_test, y_test = create_sequences(X_test_scaled, y_test_scaled, n_input, n_output)
    
    return X_train, y_train, X_test, y_test, scaler, target_scaler, X_train_scaled, X_test_scaled

X_train, y_train, X_test, y_test, scaler, target_scaler, X_train_scaled, X_test_scaled = prepare_data(df, test_size)

# Verify Shapes
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# Define function for building model
In this model architecture, a one-dimensional convolutional neural network (1D CNN) is used to read and encode the input sequence. An LSTM network is then used as a decoder to make ode-step prediction for each value in the output sequence.

In [8]:
# build and train the model
def build_model(X_train_data, y_train_data, X_val_data, y_val_data):

    n_input = X_train_data.shape[1]
    n_features = X_train_data.shape[2]
    
    model = Sequential([
        #-------CNN-------
        Conv1D(64, 5, activation='relu', padding='same', input_shape=(n_input, n_features)),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        #-------CNN-------
        #-------LSTM-------
        LSTM(64, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
        Dense(1)
        #-------LSTM-------
    ])


    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005))
    model.summary()
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

    # fit network and save the history
    training_history = model.fit(
        X_train_data, y_train_data, 
        epochs=150, batch_size= 128, 
        verbose= 1, 
        validation_data=(X_val_data, y_val_data), 
        callbacks = [early_stop, reduce_lr])
    return model, training_history

# Define function for evaluating model

# Train model and make predictions

In [ ]:
%%time

# Clear old model from memory
tf.keras.backend.clear_session()

X_train, y_train, X_test, y_test, scaler, target_scaler, X_train_scaled, X_test_scaled = prepare_data(df, test_size)


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# Split validation 
val_size = int(len(X_train) * 0.2)
X_tr, X_val = X_train[:-val_size], X_train[-val_size:]
y_tr, y_val = y_train[:-val_size], y_train[-val_size:]

# Train model 
model, training_history = build_model(X_tr, y_tr, X_val, y_val)

print("\nGenerating predictions...")
# Predict the whole test set in one go using the sequenced X_test
y_pred_scaled = model.predict(X_test)

# Inverse Transform to get real Degrees Celsius back
predictions_celsius = target_scaler.inverse_transform(y_pred_scaled)
actual_celsius = target_scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = np.sqrt(mean_squared_error(actual_celsius, predictions_celsius))
mae = mean_absolute_error(actual_celsius, predictions_celsius)
r2 = r2_score(actual_celsius, predictions_celsius)

print("\n--- FINAL RESULTS ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} °C")
print(f"Mean Absolute Error (MAE): {mae:.4f} °C")
print(f"R-squared Score (R2): {r2:.4f}")

# Plot the Results
plt.figure(figsize=(14, 6))
plt.plot(actual_celsius[:500], label='Actual Temperature (°C)', color='blue', linewidth=1.5)
plt.plot(predictions_celsius[:500], label='Predicted Temperature (°C)', color='orange', linewidth=1.5)
plt.title('HVAC Return Air Temperature: Actual vs Predicted')
plt.xlabel('Time Steps')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True)
plt.show()

# Visualize model architecture

In [ ]:
from IPython.display import Image, display
from tensorflow.keras.utils import plot_model

# display the model architecture
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
display(Image(filename='model_architecture.png'))

# Compare actual and predicted values

In [ ]:
min_len = min(len(actual_celsius), len(predictions_celsius))
final_actual = actual_celsius[:min_len]
final_pred = predictions_celsius[:min_len]

def plot_results(actual, predicted):
    plt.figure(figsize=(20, 8))
    plt.plot(actual, label='Actual Temperature', color='blue', linewidth=1.5)
    plt.plot(predicted, label='Predicted Temperature', color='orange', alpha=0.8, linewidth=1.5)
    plt.title('Final Results: Actual vs Predicted Inlet Temperature')
    plt.ylabel('Temperature (°C)')
    plt.xlabel('Time Steps (Hours)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Run the plot with the Celsius variables
plot_results(final_actual, final_pred)

from sklearn.metrics import mean_absolute_error, mean_squared_error
print(f"MAE: {mean_absolute_error(final_actual, final_pred):.4f} °C")
print(f"RMSE: {np.sqrt(mean_squared_error(final_actual, final_pred)):.4f} °C")
print(f"R2 Score: {r2_score(final_actual, final_pred):.4f}")

# RMSE for all sequences

In [ ]:
time_end = time.time()
print("Notebook run time: {:.0f} seconds".format(time_end - time_start))

## Performance metric for the baseline model (CNN-LSTM) and Models training loss and validation loss

In [ ]:
def return_performance_metrics(actual_vals, predicted_vals):
    rmse = np.sqrt(mean_squared_error(actual_vals, predicted_vals))
    mae = mean_absolute_error(actual_vals, predicted_vals)
    r2 = r2_score(actual_vals, predicted_vals)
    
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f} °C")
    print(f"Mean Absolute Error (MAE): {mae:.4f} °C")
    print(f"R-squared Score (R2): {r2:.4f}")
    return rmse, mae, r2

# Use the variables we aligned in the previous cell
# final_actual and final_pred are the Celsius versions
metrics = return_performance_metrics(final_actual, final_pred)

plt.figure(figsize=(10, 6))
# Using the global training_history object
plt.plot(training_history.history['loss'], label='Train Loss (MSE)', color='blue')
plt.plot(training_history.history['val_loss'], label='Validation Loss (MSE)', color='red')
plt.title('Model Learning Curve: Loss During Training')
plt.ylabel('Loss (Mean Squared Error)')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# Save model and config for DDPG training

In [ ]:
import pickle

# Save CNN-LSTM model
model.save('cnn_lstm_hvac.keras')

# Save scalers
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

# Save feature columns and config
config = {
    'FEATURE_COLS': FEATURE_COLS,
    'TARGET_COL': TARGET_COL,
    'n_input': n_input,
    'n_output': n_output
}
with open('model_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("Model and scalers saved!")
print(f"Feature columns: {FEATURE_COLS}")
print(f"n_input: {n_input}")